# Spindle — Streaming & Anomaly Injection

Spindle can stream synthetic events row-by-row with realistic arrival patterns — ideal for feeding **Microsoft Fabric Eventstream**, Azure Event Hubs, or Kafka topics.

**Features**
- Poisson inter-arrival times (statistically realistic pacing)
- Token-bucket rate limiting with configurable burst windows
- Out-of-order event injection (simulates late-arriving data)
- Three anomaly types: point, contextual, collective
- Sinks: `ConsoleSink`, `FileSink` (built-in); `EventHubSink`, `KafkaSink` (`pip install sqllocks-spindle[streaming]`)

**Sinks**
| Sink | Install | Use case |
|---|---|---|
| `ConsoleSink` | built-in | Development / debugging |
| `FileSink` | built-in | Local JSONL files |
| `EventHubSink` | `[streaming]` | Fabric Eventstream, Azure Event Hubs |
| `KafkaSink` | `[streaming]` | Apache Kafka |


In [1]:
%pip install sqllocks-spindle --quiet
# For Event Hub / Kafka support:
# %pip install sqllocks-spindle[streaming] --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


## Basic streaming — console sink

`realtime=False` emits events as fast as possible (best for Fabric notebooks).
Set `realtime=True` to enable Poisson inter-arrivals + token-bucket rate limiting.

In [2]:
from sqllocks_spindle import RetailDomain
from sqllocks_spindle.streaming import SpindleStreamer, StreamConfig, ConsoleSink

result = SpindleStreamer(
    domain=RetailDomain(),
    sink=ConsoleSink(indent=2),              # pretty-print with 2-space indent
    config=StreamConfig(max_events=20, realtime=False),
    scale="fabric_demo",
    seed=42,
).stream("order")

print(result)

{
  "order_id": 29,
  "customer_id": 130,
  "store_id": 2,
  "shipping_address_id": 67,
  "promotion_id": null,
  "order_date": "2022-03-17 09:39:19.113926432",
  "status": "completed",
  "order_total": 19.48,
  "_spindle_table": "order",
  "_spindle_seq": 28,
  "_spindle_event_time": "2022-03-17 09:39:19.113926432"
}
{
  "order_id": 897,
  "customer_id": 31,
  "store_id": 5,
  "shipping_address_id": 8,
  "promotion_id": null,
  "order_date": "2022-04-25 02:05:54",
  "status": "completed",
  "order_total": 123.95,
  "_spindle_table": "order",
  "_spindle_seq": 896,
  "_spindle_event_time": "2022-04-25 02:05:54"
}
{
  "order_id": 526,
  "customer_id": 142,
  "store_id": 2,
  "shipping_address_id": null,
  "promotion_id": null,
  "order_date": "2022-04-26 19:47:58",
  "status": "completed",
  "order_total": 16.21,
  "_spindle_table": "order",
  "_spindle_seq": 525,
  "_spindle_event_time": "2022-04-26 19:47:58"
}
{
  "order_id": 561,
  "customer_id": 26,
  "store_id": 1,
  "shipping_addr

## File sink — JSONL output

Write events to a JSONL file for inspection or downstream processing.

In [3]:
import tempfile, os, json
from sqllocks_spindle.streaming import FileSink

with tempfile.TemporaryDirectory() as tmpdir:
    outfile = os.path.join(tmpdir, "orders.jsonl")

    result = SpindleStreamer(
        domain=RetailDomain(),
        sink=FileSink(outfile, mode="w"),
        config=StreamConfig(max_events=50, realtime=False),
        scale="fabric_demo",
        seed=42,
    ).stream("order")

    print(result)

    # Inspect a few lines
    with open(outfile) as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(json.loads(line))

StreamResult(table='order', events=50, anomalies=0, ooo=0, 0.00s @ 148,418 eps)
{'order_id': 29, 'customer_id': 130, 'store_id': 2, 'shipping_address_id': 67, 'promotion_id': None, 'order_date': '2022-03-17 09:39:19.113926432', 'status': 'completed', 'order_total': 19.48, '_spindle_table': 'order', '_spindle_seq': 28, '_spindle_event_time': '2022-03-17 09:39:19.113926432'}
{'order_id': 897, 'customer_id': 31, 'store_id': 5, 'shipping_address_id': 8, 'promotion_id': None, 'order_date': '2022-04-25 02:05:54', 'status': 'completed', 'order_total': 123.95, '_spindle_table': 'order', '_spindle_seq': 896, '_spindle_event_time': '2022-04-25 02:05:54'}
{'order_id': 526, 'customer_id': 142, 'store_id': 2, 'shipping_address_id': None, 'promotion_id': None, 'order_date': '2022-04-26 19:47:58', 'status': 'completed', 'order_total': 16.21, '_spindle_table': 'order', '_spindle_seq': 525, '_spindle_event_time': '2022-04-26 19:47:58'}


## Anomaly injection

Inject synthetic anomalies into the stream. All anomalous events are tagged with `_spindle_is_anomaly=True` and `_spindle_anomaly_type`.

In [4]:
from sqllocks_spindle.streaming import AnomalyRegistry, PointAnomaly, ContextualAnomaly

registry = AnomalyRegistry([
    # 5% of events: order_total spiked to 10x normal range
    PointAnomaly(name="extreme_total", column="order_total", multiplier_range=(10.0, 10.0), fraction=0.05),

    # Cancelled orders sometimes still have high totals (contextual)
    ContextualAnomaly(
        name="cancelled_high_total",
        column="order_total",
        condition_column="status",
        normal_values=["cancelled"],
        anomalous_values=[500.0, 1000.0, 1500.0, 2000.0],
        fraction=0.30,
    ),
])

with tempfile.TemporaryDirectory() as tmpdir:
    outfile = os.path.join(tmpdir, "orders_with_anomalies.jsonl")

    result = SpindleStreamer(
        domain=RetailDomain(),
        sink=FileSink(outfile, mode="w"),
        config=StreamConfig(max_events=200, realtime=False),
        anomaly_registry=registry,
        scale="fabric_demo",
        seed=42,
    ).stream("order")

    print(result)
    print(f"Anomalies injected: {result.anomaly_count} / {result.events_sent} ({result.anomaly_count/result.events_sent:.1%})")

StreamResult(table='order', events=200, anomalies=58, ooo=0, 0.00s @ 237,100 eps)
Anomalies injected: 58 / 200 (29.0%)


## Burst windows

Simulate traffic spikes — e.g. flash sales, Black Friday, end-of-month billing.

`BurstWindow(start_offset_seconds, duration_seconds, multiplier)` — after `start_offset_seconds` seconds, emit at `multiplier`× the base rate for `duration_seconds` seconds.

In [5]:
from sqllocks_spindle.streaming import BurstWindow

config = StreamConfig(
    events_per_second=50.0,  # 50 events/sec baseline
    realtime=True,           # enable rate limiting
    max_events=150,
    burst_windows=[
        BurstWindow(start_offset_seconds=1.0, duration_seconds=2.0, multiplier=5.0),  # 5× spike at t=1s
    ],
)

print("BurstWindow config ready.")
print(f"  Baseline rate: {config.events_per_second} events/sec")
print(f"  Burst at t=1s: 5× for 2 seconds ({config.events_per_second * 5:.0f} events/sec)")
print()
print("Run SpindleStreamer with this config to observe burst behavior.")
print("(Omitting live run here to avoid blocking the notebook for 3+ seconds.)")

BurstWindow config ready.
  Baseline rate: 50.0 events/sec
  Burst at t=1s: 5× for 2 seconds (250 events/sec)

Run SpindleStreamer with this config to observe burst behavior.
(Omitting live run here to avoid blocking the notebook for 3+ seconds.)


## Event Hub sink — Fabric Eventstream

Requires `pip install sqllocks-spindle[streaming]` and an Azure Event Hubs connection string.

In Fabric, create an **Eventstream** → add a **Custom App** source → copy the connection string.

In [6]:
# Uncomment and fill in your connection string to run

# from sqllocks_spindle.streaming import EventHubSink
#
# CONNECTION_STRING = "Endpoint=sb://..."  # from Fabric Eventstream Custom App source
# EVENTHUB_NAME    = "spindle-retail"
#
# result = SpindleStreamer(
#     domain=RetailDomain(),
#     sink=EventHubSink(CONNECTION_STRING, eventhub_name=EVENTHUB_NAME),
#     config=StreamConfig(events_per_second=100.0, realtime=True, max_events=1000),
#     scale="fabric_demo",
#     seed=42,
# ).stream("order")
#
# print(result)

print("Fill in CONNECTION_STRING and EVENTHUB_NAME above, then uncomment to run.")

Fill in CONNECTION_STRING and EVENTHUB_NAME above, then uncomment to run.


## Stream multiple tables

Run a streamer per table to simulate concurrent event streams.

In [7]:
from sqllocks_spindle import Spindle

# Pre-generate tables once, then stream each independently
spindle = Spindle()
batch = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)

for table_name in ["order", "order_line", "return"]:
    with tempfile.TemporaryDirectory() as tmpdir:
        r = SpindleStreamer(
            tables=batch.tables,
            sink=FileSink(os.path.join(tmpdir, f"{table_name}.jsonl"), mode="w"),
            config=StreamConfig(max_events=30, realtime=False),
        ).stream(table_name)
        print(f"  {table_name:<12} {r}")

  order        StreamResult(table='order', events=30, anomalies=0, ooo=0, 0.00s @ 13,597 eps)
  order_line   StreamResult(table='order_line', events=30, anomalies=0, ooo=0, 0.00s @ 140,591 eps)
  return       StreamResult(table='return', events=30, anomalies=0, ooo=0, 0.00s @ 174,279 eps)
